#  Force Hugging Face to fall back to the standard HTTP download



In [ ]:
import os
os.environ["HF_HUB_DISABLE_XET"] = "1"

# Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


# Pull Repo

In [ ]:
%cd /content/drive/MyDrive

import os
if os.path.isdir("soc-agent"):
    %cd soc-agent
    !git pull origin main
else:
    !git clone https://github.com/avidzcheetah/soc-agent.git
    %cd soc-agent


/content/drive/MyDrive
/content/drive/MyDrive/soc-agent
remote: Enumerating objects: 32, done.
remote: Counting objects: 100% (32/32), done.
remote: Compressing objects: 100% (20/20), done.
remote: Total 26 (delta 13), reused 19 (delta 6), pack-reused 0 (from 0)
Unpacking objects: 100% (26/26), 13.74 KiB | 2.00 KiB/s, done.
From https://github.com/avidzcheetah/soc-agent
 * branch            main       -> FETCH_HEAD
   fa7024e..3c78aa3  main       -> origin/main
Updating fa7024e..3c78aa3
Fast-forward
 docs/experiment_index.md         |   5 +-
 docs/secbert_training_journey.md |  17 +++-
 docs/thesis_notes.md             |   5 +
 notebooks/demo_inference.ipynb   |  65 +++++++++++++
 scripts/demo_inference.py        |  83 ++++++++++++++++
 scripts/generate_plots.py        | 199 +++++++++++++++++++++++++++++++++++++++
 6 files changed, 367 insertions(+), 7 deletions(-)
 create mode 100644 notebooks/demo_inference.ipynb
 create mode 100644 scripts/demo_inference.py
 create mode 100644 scrip

# Install Dependancies

In [ ]:
%cd /content/drive/MyDrive/soc-agent

!pip install -q -r requirements.txt
!nvidia-smi


/content/drive/MyDrive/soc-agent
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 187.6/187.6 kB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.0 MB/s eta 0:00:00
/bin/bash: line 1: nvidia-smi: command not found


# Train SecBERT

In [ ]:
!PYTHONPATH=. python src/train_secbert.py --config configs/secbert.yaml --device cuda

2026-07-14 09:39:20.018830: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
Environment Verification
Python       : 3.12.13
Torch        : 2.11.0+cu128
CUDA         : Available
GPU          : Tesla T4
VRAM         : 14.56 GB
Transformers : 5.12.1
Datasets     : 4.0.0
Accelerate   : 1.14.0
Device       : cuda
Initialized Experiment: EXP_20260714_003
Saved initial experiment.json to experiments/EXP_20260714_003/metadata/experiment.json
Dataset Verification
Sizes:
  [PASS] Train split non-empty         :  12512 records
  [PASS] Val split non-empty           :   1539 records
  [PASS] Test split non-empty          :   1539 records
Columns:
  [PASS] 'text' column present in train
  [PASS] 'action_label' column present in train
  [PASS] 'text' column pr

#Train with full Precision

In [ ]:
!PYTHONPATH=. python src/train_secbert.py --config configs/secbert_fp32.yaml --device cuda

2026-07-27 07:21:57.729625: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
Environment Verification
Python       : 3.12.13
Torch        : 2.11.0+cu128
CUDA         : Available
GPU          : Tesla T4
VRAM         : 14.56 GB
Transformers : 5.13.1
Datasets     : 4.0.0
Accelerate   : 1.14.0
Device       : cuda
Initialized Experiment: EXP_20260727_001
Saved initial experiment.json to experiments/EXP_20260727_001/metadata/experiment.json
Dataset Verification
Sizes:
  [PASS] Train split non-empty         :  12512 records
  [PASS] Val split non-empty           :   1539 records
  [PASS] Test split non-empty          :   1539 records
Columns:
  [PASS] 'text' column present in train
  [PASS] 'action_label' column present in train
  [PASS] 'text' column pr

# Model Demonstration

In [ ]:
import os
import torch
import pandas as pd
from transformers import AutoModelForSequenceClassification, AutoTokenizer

def load_label_map(csv_path='data/action_space.csv'):
    df = pd.read_csv(csv_path)
    return dict(zip(df['Index'], df['Action Name']))

def predict(text, model, tokenizer, label_map, device='cpu'):
    inputs = tokenizer(text, return_tensors='pt', truncation=True, max_length=512).to(device)
    with torch.no_grad():
        probs = torch.nn.functional.softmax(model(**inputs).logits, dim=-1)[0]

    top_probs, top_indices = torch.topk(probs, 3)
    print(f'Input Text: {text}')
    print('-' * 50)
    for i in range(3):
        prob = top_probs[i].item()
        label = label_map.get(top_indices[i].item(), 'Unknown')
        print(f'{i+1}. {label} ({prob:.2%})')
    print('=' * 50 + '\n')

model_path = 'experiments/EXP_20260713_001/checkpoints/best_model'
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

if os.path.exists(model_path):
    model = AutoModelForSequenceClassification.from_pretrained(model_path).to(device)
    tokenizer = AutoTokenizer.from_pretrained(model_path)
    model.eval()
    label_map = load_label_map('data/action_space.csv')
    print(f'Model loaded successfully on {device}!\n')

    samples = [
        'Multiple failed login attempts detected from IP 192.168.1.55 targeting the admin account via SSH.',
        'An unauthorized process mimikatz.exe was blocked from executing on workstation-04 by the EDR.',
    ]

    for s in samples:
        predict(s, model, tokenizer, label_map, device)
else:
    print(f'Error: Model not found at {model_path}')


Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

Model loaded successfully on cpu!

Input Text: Multiple failed login attempts detected from IP 192.168.1.55 targeting the admin account via SSH.
--------------------------------------------------
1. block_port (46.70%)
2. snapshot_forensics (18.02%)
3. restore_defense_config (3.77%)

Input Text: An unauthorized process mimikatz.exe was blocked from executing on workstation-04 by the EDR.
--------------------------------------------------
1. snapshot_forensics (26.34%)
2. disable_account (20.01%)
3. kill_process (6.22%)

